
# Pokémon Card Market Analytics & Price Prediction
## Professional Data Science Project

This notebook contains a complete end-to-end analysis of the Pokémon card e-commerce marketplace dataset.

### Covered Areas
- Data Cleaning & Preprocessing
- Exploratory Data Analysis (EDA)
- Missing Value Analysis
- Outlier Detection
- Feature Engineering
- Price Trend Analysis
- Rarity & Value Insights
- Correlation Analysis
- Seller Behavior Analysis
- Advanced Visualizations
- Predictive Machine Learning Models
- Feature Importance Analysis
- Business Insights & Recommendations

---

## Dataset Overview
- Rows: **542**
- Columns: **32**



In [ ]:

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


In [ ]:

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv(r"/mnt/data/pokemon_cards_ultimate_2026.csv")

print("Dataset Shape:", df.shape)

df.head()


## Dataset Information & Initial Inspection

In [ ]:

# Dataset Information
df.info()


In [ ]:

# Statistical Summary
df.describe(include='all').T


## Missing Value Analysis

In [ ]:

# Missing Values
missing = df.isnull().sum().sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Values': missing.values,
    'Missing Percentage': (missing.values / len(df)) * 100
})

missing_df.head(20)


In [ ]:

# Visualize Missing Values

plt.figure(figsize=(14,6))
sns.barplot(
    x=missing_df['Column'][:20],
    y=missing_df['Missing Percentage'][:20]
)

plt.xticks(rotation=90)
plt.title("Top Missing Value Percentages")
plt.ylabel("Missing %")
plt.show()


## Data Cleaning & Preprocessing

In [ ]:

# Duplicate Records
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

# Remove duplicates if needed
df = df.drop_duplicates()

print("Shape After Deduplication:", df.shape)


In [ ]:

# Convert numerical columns where possible

numeric_candidates = [
    'price',
    'price_usd',
    'numeric_grade',
    'seller_listing_count',
    'image_count',
    'days_since_sold',
    'sale_month',
    'sale_year'
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Numeric conversion completed.")


## Exploratory Data Analysis (EDA)

In [ ]:

# Target Variable Distribution

plt.figure(figsize=(12,6))

sns.histplot(df['price_usd'].dropna(), bins=50, kde=True)

plt.title("Distribution of Pokémon Card Prices (USD)")
plt.xlabel("Price USD")
plt.ylabel("Frequency")

plt.show()


In [ ]:

# Log Distribution (important for skewed prices)

plt.figure(figsize=(12,6))

sns.histplot(np.log1p(df['price_usd'].dropna()), bins=50, kde=True)

plt.title("Log Distribution of Pokémon Card Prices")
plt.xlabel("Log Price USD")

plt.show()


## Rarity & Price Insights

In [ ]:

# Average Price by Rarity

rarity_price = (
    df.groupby('rarity_class')['price_usd']
    .mean()
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(14,6))

sns.barplot(
    x=rarity_price.index,
    y=rarity_price.values
)

plt.xticks(rotation=45)
plt.title("Average Price by Rarity")
plt.ylabel("Average Price USD")

plt.show()

rarity_price


In [ ]:

# Premium Card Feature Analysis

special_cols = [
    'is_holo',
    'is_full_art',
    'is_v_card',
    'is_ex_card',
    'is_gx_card',
    'is_promo',
    'is_shadowless',
    'is_1st_edition',
    'is_rainbow',
    'is_gold'
]

available_special_cols = [c for c in special_cols if c in df.columns]

premium_analysis = {}

for col in available_special_cols:
    premium_analysis[col] = (
        df.groupby(col)['price_usd']
        .mean()
        .to_dict()
    )

premium_analysis


## Language & Market Analysis

In [ ]:

# Language Distribution

plt.figure(figsize=(10,5))

df['language'].value_counts().head(10).plot(kind='bar')

plt.title("Top Languages")
plt.ylabel("Count")

plt.show()


In [ ]:

# Average Price by Language

lang_price = (
    df.groupby('language')['price_usd']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(12,5))

sns.barplot(
    x=lang_price.index,
    y=lang_price.values
)

plt.title("Average Price by Language")
plt.ylabel("Average Price USD")

plt.xticks(rotation=45)

plt.show()

lang_price


## Seller Behavior Analysis

In [ ]:

# Top Seller Countries

top_countries = df['seller_country'].value_counts().head(15)

plt.figure(figsize=(14,6))

sns.barplot(
    x=top_countries.index,
    y=top_countries.values
)

plt.xticks(rotation=45)
plt.title("Top Seller Countries")

plt.show()


In [ ]:

# Seller Listing Count vs Price

if 'seller_listing_count' in df.columns:

    plt.figure(figsize=(10,6))

    sns.scatterplot(
        x=df['seller_listing_count'],
        y=df['price_usd']
    )

    plt.title("Seller Listing Count vs Price")

    plt.show()


## Outlier Detection

In [ ]:

# Boxplot for Price Outliers

plt.figure(figsize=(14,5))

sns.boxplot(x=df['price_usd'])

plt.title("Price Outlier Detection")

plt.show()


In [ ]:

# IQR-Based Outlier Detection

Q1 = df['price_usd'].quantile(0.25)
Q3 = df['price_usd'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['price_usd'] < lower_bound) |
    (df['price_usd'] > upper_bound)
]

print("Number of Outliers:", len(outliers))

outliers.head()


## Correlation Analysis

In [ ]:

# Correlation Matrix

numeric_df = df.select_dtypes(include=np.number)

corr = numeric_df.corr()

plt.figure(figsize=(16,10))

sns.heatmap(
    corr,
    cmap='coolwarm',
    annot=False
)

plt.title("Correlation Matrix")

plt.show()


## Feature Engineering

In [ ]:

# Feature Engineering

# Premium Card Indicator
premium_features = [
    'is_holo',
    'is_full_art',
    'is_gold',
    'is_rainbow',
    'is_1st_edition'
]

available = [c for c in premium_features if c in df.columns]

df['premium_score'] = df[available].sum(axis=1)

# Price Per Image
if 'image_count' in df.columns:
    df['price_per_image'] = (
        df['price_usd'] /
        df['image_count'].replace(0, np.nan)
    )

df[['premium_score']].head()


## Price Trend Analysis

In [ ]:

# Monthly Price Trends

if 'sale_month' in df.columns and 'price_usd' in df.columns:

    monthly_trend = (
        df.groupby('sale_month')['price_usd']
        .mean()
    )

    plt.figure(figsize=(12,5))

    monthly_trend.plot(marker='o')

    plt.title("Average Monthly Pokémon Card Prices")
    plt.ylabel("Average Price USD")

    plt.show()

    monthly_trend


## Predictive Machine Learning Model

In [ ]:

# =========================
# MACHINE LEARNING
# =========================

target = 'price_usd'

feature_candidates = [
    'rarity_class',
    'language',
    'condition_std',
    'seller_country',
    'is_holo',
    'is_full_art',
    'is_v_card',
    'is_ex_card',
    'is_gx_card',
    'is_promo',
    'is_shadowless',
    'is_1st_edition',
    'is_rainbow',
    'is_gold',
    'premium_score',
    'seller_listing_count',
    'image_count'
]

features = [f for f in feature_candidates if f in df.columns]

X = df[features]
y = df[target]

categorical_features = [
    c for c in X.columns
    if X[c].dtype == 'object'
]

numeric_features = [
    c for c in X.columns
    if c not in categorical_features
]

# Preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train Model
model.fit(X_train, y_train)

# Predictions
preds = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))


## Feature Importance

In [ ]:

# Feature Importance

rf_model = model.named_steps['model']

encoded_cat = model.named_steps['preprocessor']\
    .named_transformers_['cat']\
    .named_steps['encoder']\
    .get_feature_names_out(categorical_features)

all_features = numeric_features + list(encoded_cat)

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

top_features = importance_df.head(20)

plt.figure(figsize=(12,8))

sns.barplot(
    x='Importance',
    y='Feature',
    data=top_features
)

plt.title("Top 20 Feature Importances")

plt.show()

top_features



# Business Insights & Recommendations

## Key Market Insights

### 1. Rarity Drives Premium Pricing
High-rarity cards such as SAR, UR, Secret Rare, and alternate arts generally command higher prices.

### 2. Premium Attributes Increase Value
Features such as:
- Holographic finish
- Full Art
- Rainbow
- Gold
- 1st Edition
- Shadowless

typically increase market value substantially.

### 3. Language Demand Matters
Japanese cards often outperform English cards in collectible demand due to:
- Exclusive artwork
- Print quality
- Earlier release cycles

### 4. Seller Optimization Opportunities
Listings with:
- More images
- Better condition transparency
- International shipping

can improve buyer trust and increase sale prices.

### 5. Grading Potential
PSA/BGS grading can dramatically increase card value for:
- Vintage cards
- Mint condition cards
- Rare promotional releases

---

# Strategic Recommendations

## For Collectors
- Focus on high-rarity Japanese cards
- Prioritize mint-condition collectibles
- Watch undervalued premium cards

## For Sellers
- Improve listing quality
- Add more product images
- Offer global shipping

## For Businesses
- Build automated price prediction systems
- Create rarity scoring engines
- Track market trend movements over time



# Conclusion

This notebook demonstrates a complete professional-grade Pokémon card market analysis pipeline.

The workflow includes:
- Data cleaning
- Advanced exploratory analysis
- Feature engineering
- Predictive machine learning
- Market intelligence generation

With larger datasets and real-time market feeds, this framework can scale into:
- Marketplace analytics platforms
- AI-powered pricing engines
- Investment tracking systems
- Card recommendation engines
